# Sample 47 — Runtime demo: create and invoke the agent

Terraform (roots `01`–`07`) has laid down the platform: hub Foundry account + APIM + policy allowlist, spoke Foundry project + capability host + `gateway` connection, and the Application Gateway shell wired to expose the agent endpoint publicly. This notebook does the **runtime** step: create an agent on the spoke project, list it, invoke it, and show the APIM quota headers and negative-path enforcement.

## Where to run this

**On a jumpbox VM inside the spoke VNet**, reached via Bastion. This is required because the spoke Foundry project account has `publicNetworkAccess = Disabled` — the data-plane hostname `*.services.ai.azure.com` only resolves and connects from a network that is linked to the private DNS zones and peered to the hub. The demo notebook cannot run from a laptop over public internet.

Prereqs on the jumpbox:

- Azure CLI installed (`az --version`) and logged in (`az login`).
- `curl` and `jq`.
- The signed-in user has **Azure AI Developer** (or Cognitive Services User) role on the spoke Foundry project — required to author agents on the data plane. If you skipped this, run `az role assignment create` before proceeding.
- Python + `ipykernel` if you want to run this file as a notebook; otherwise the same cells copy-paste cleanly into a shell.

## What the notebook proves

1. **Agent creation succeeds against the spoke project** — data-plane call, private DNS resolution, works.
2. **The agent's model resolves through APIM**, not through a local Foundry deployment — proving the BYOM story.
3. **The APIM quota policy is engaged** — `x-aigw-tokens-consumed` / `x-aigw-tokens-remaining` headers land on the response.
4. **Negative paths are enforced** — the `oid` allowlist rejects unknown callers (401) and the alias `otherwise` branch rejects unknown model names (404).

## 1. Prereq — set demo variables

Fill these in from your Terraform outputs and tfvars. Use `terraform output -raw <name>` if you want to script it.

In [ ]:
# --- IDs (copy from your terraform.tfvars / terraform output) ------------
export SPOKE_SUBSCRIPTION_ID="b2bba6b4-c67d-43f2-9fbf-998560a9bdb0"
export HUB_SUBSCRIPTION_ID="8861a79b-122e-4733-b9f0-bb521b0268ce"
export TENANT_ID="e6f8ac30-2fd5-4173-8392-31b071846feb"

# --- Spoke Foundry project (from 05-spoke-workload-project outputs) ------
export FOUNDRY_ACCOUNT="foundrysample-byom-lob-hr-fdry-2472w"
export PROJECT_NAME="hrbot"
# services.ai.azure.com is the current data-plane host for Foundry.
# If the Learn docs move the endpoint, adjust here.
export PROJECT_ENDPOINT="https://${FOUNDRY_ACCOUNT}.services.ai.azure.com/api/projects/${PROJECT_NAME}"

# --- APIM (from 04-hub-workload outputs) ---------------------------------
export APIM_PRIVATE_URL="https://foundrysample-byom-apim-25dyf.privatelink.azure-api.net/inference"

# --- Catalog (from 04-hub-workload outputs) ------------------------------
export MODEL_ALIAS="gpt-4.1"
export CONNECTION_NAME="gateway"          # the APIM connection created in step 05

# --- API versions --------------------------------------------------------
# Check https://learn.microsoft.com/azure/ai-foundry/ for the current preview.
# Foundry Agents REST (v1 GA) — see
# https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/prompt-agent?tabs=rest
export AGENTS_API_VERSION="v1"
export INFERENCE_API_VERSION="2024-10-21"

echo "PROJECT_ENDPOINT = $PROJECT_ENDPOINT"
echo "APIM_PRIVATE_URL = $APIM_PRIVATE_URL"

## 2. Get tokens

Two audiences:

- `AI_TOKEN` — data plane at `https://ai.azure.com`. Used to author the agent on the project.
- `APIM_TOKEN` — audience `https://cognitiveservices.azure.com`. Used to hit APIM directly for the header / negative tests. **Your `oid` is NOT in the allowlist** — that is the point of the 401 negative test in §6.

In [ ]:
az account set --subscription "$SPOKE_SUBSCRIPTION_ID"

export AI_TOKEN=$(az account get-access-token --resource "https://ai.azure.com" --query accessToken -o tsv)
export APIM_TOKEN=$(az account get-access-token --resource "https://cognitiveservices.azure.com" --query accessToken -o tsv)

echo "AI_TOKEN length:   ${#AI_TOKEN}"
echo "APIM_TOKEN length: ${#APIM_TOKEN}"

## 3. Create the agent

The `model` field references the **APIM connection** (`gateway`) and the catalog **alias** (`gpt-4.1`). Foundry treats `connection/alias` as "resolve model calls through this connection, using this alias as the model name." The connection's target URL is the APIM private FQDN, and its auth is the project MI — so every model call generated by this agent gets a token from the project MI and lands on APIM with the allowlisted `oid` claim.

> **API shape caveat.** The persistent-agents authoring API has moved a few times. If this call 400s with `"unknown field"` or similar, check the Learn docs for the current body shape and adjust the JSON below — the intent (bind model to a connection + alias) is stable, the field names are not.

In [ ]:
curl -sS -X POST \
  "/agents?api-version=" \
  -H "Authorization: Bearer " \
  -H "Content-Type: application/json" \
  -d @- <<JSON | jq .
{
  "name": "hr-assistant",
  "description": "Sample 47 demo agent bound to the platform APIM catalog.",
  "definition": {
    "kind": "prompt",
    "model": "${CONNECTION_NAME}/${MODEL_ALIAS}",
    "instructions": "You answer HR policy questions concisely. If unsure, say so."
  }
}
JSON

Capture the agent id for later cells:

In [ ]:
export AGENT_ID=$(curl -sS \
  "${PROJECT_ENDPOINT}/agents?api-version=${AGENTS_API_VERSION}" \
  -H "Authorization: Bearer ${AI_TOKEN}" \
  | jq -r '.data[] | select(.name=="hr-assistant") | .id')

echo "AGENT_ID=$AGENT_ID"

## 4. List agents

Sanity check — the project now owns exactly one agent, bound to the gateway connection.

In [ ]:
curl -sS \
  "${PROJECT_ENDPOINT}/agents?api-version=${AGENTS_API_VERSION}" \
  -H "Authorization: Bearer ${AI_TOKEN}" \
  | jq '.data[] | {id, name, model}'

## 5. Create a thread + run — full end-to-end invocation

This is the moment of truth. The run causes Foundry to:

1. Resolve `gateway/gpt-4.1` via the project's `gateway` connection.
2. Acquire a token as the **project MI** for audience `https://cognitiveservices.azure.com`.
3. POST `/inference/deployments/gpt-4.1/chat/completions?api-version=...` to the APIM private URL.
4. APIM's inbound policy verifies the JWT and the `oid` claim against the allowlist (the project MI OID **is** allowlisted).
5. APIM's `choose` block dispatches to the Foundry backend via `authentication-managed-identity` and the response streams back.

The `thread.run` call itself is data-plane on the project.

In [ ]:
# Single-shot: create a thread with an initial message and start a run.
export RUN_RESPONSE=$(curl -sS -X POST \
  "${PROJECT_ENDPOINT}/threads/runs?api-version=${AGENTS_API_VERSION}" \
  -H "Authorization: Bearer ${AI_TOKEN}" \
  -H "Content-Type: application/json" \
  -d @- <<JSON
{
  "assistant_id": "${AGENT_ID}",
  "thread": {
    "messages": [
      {"role": "user", "content": "In two sentences, what is a good workday for an HR generalist?"}
    ]
  }
}
JSON
)

echo "$RUN_RESPONSE" | jq .

export THREAD_ID=$(echo "$RUN_RESPONSE" | jq -r .thread_id)
export RUN_ID=$(echo "$RUN_RESPONSE" | jq -r .id)
echo "THREAD_ID=$THREAD_ID  RUN_ID=$RUN_ID"

Poll the run status until it completes:

In [ ]:
for i in 1 2 3 4 5 6 7 8 9 10; do
  STATUS=$(curl -sS \
    "${PROJECT_ENDPOINT}/threads/${THREAD_ID}/runs/${RUN_ID}?api-version=${AGENTS_API_VERSION}" \
    -H "Authorization: Bearer ${AI_TOKEN}" \
    | jq -r .status)
  echo "attempt $i: $STATUS"
  if [ "$STATUS" = "completed" ] || [ "$STATUS" = "failed" ] || [ "$STATUS" = "cancelled" ]; then
    break
  fi
  sleep 3
done

# Fetch the last assistant message.
curl -sS \
  "${PROJECT_ENDPOINT}/threads/${THREAD_ID}/messages?api-version=${AGENTS_API_VERSION}" \
  -H "Authorization: Bearer ${AI_TOKEN}" \
  | jq '.data[] | select(.role=="assistant") | .content'

## 6. Show APIM quota headers on a direct chat/completions call

> **Note.** Your Azure CLI `oid` is not in the allowlist — this call will 401 unless you temporarily add your `oid` to `06-hub-allowlist/allowlist.auto.tfvars` and re-apply. That's the whole point of §7's negative test; the cell below is the *positive* variant you can uncomment once you've allowlisted yourself.

The `-i` flag shows the response headers. Look for `x-aigw-tokens-consumed`, `x-aigw-tokens-remaining`, `x-aigw-project-id`, `x-aigw-model-alias`, `x-aigw-vendor`.

In [ ]:
# Uncomment and run only if your oid is currently in the allowlist.
# curl -sS -i -X POST \
#   "${APIM_PRIVATE_URL}/deployments/${MODEL_ALIAS}/chat/completions?api-version=${INFERENCE_API_VERSION}" \
#   -H "Authorization: Bearer ${APIM_TOKEN}" \
#   -H "Content-Type: application/json" \
#   -d '{"messages":[{"role":"user","content":"say hi in five words"}]}' \
#   | sed -n '1,/^\r*$/p'   # print headers only

## 7. Negative tests — prove the policy is enforcing

### 7a. Wrong caller `oid` (401)

Direct call to APIM using **your** token. Your OID is not in `06-hub-allowlist/allowlist.auto.tfvars` — the `validate-azure-ad-token` gate rejects it. Expected: HTTP 401 with the error message from `apim-policy.xml.tftpl`.

In [ ]:
curl -sS -i -X POST \
  "${APIM_PRIVATE_URL}/deployments/${MODEL_ALIAS}/chat/completions?api-version=${INFERENCE_API_VERSION}" \
  -H "Authorization: Bearer ${APIM_TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{"messages":[{"role":"user","content":"hello"}]}' \
  | head -n 20

### 7b. Unknown model alias (404)

Even a fully allowlisted caller cannot request an alias outside the catalog. The `choose`/`otherwise` block returns a governance 404 with a JSON error explaining the alias is not approved.

You'll need to allowlist your `oid` first (as in §6) *or* invoke this via the agent by adding an assistant that asks for a bad model. The direct-curl variant below shows the shape.

In [ ]:
curl -sS -i -X POST \
  "${APIM_PRIVATE_URL}/deployments/no-such-alias/chat/completions?api-version=${INFERENCE_API_VERSION}" \
  -H "Authorization: Bearer ${APIM_TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{"messages":[{"role":"user","content":"hello"}]}' \
  | head -n 30

## Recap

| Layer | Enforcement demonstrated |
|---|---|
| Networking | Notebook runs from inside the peered VNet, resolves `*.services.ai.azure.com` privately, hits the Foundry PE. |
| Foundry data plane | Agent authored via `POST /agents`; `gateway/gpt-4.1` binds the model to APIM. |
| APIM inbound policy | `validate-azure-ad-token` + `oid` allowlist gates callers; `llm-token-limit` meters per-project quota. |
| APIM routing | `choose`/`when` dispatches by alias; Foundry branch uses managed-identity auth to the model account. |
| Governance | Unknown alias returns a curated 404, not a raw provider error; unknown caller returns 401. |

The IaC roots deposit *only* the substrate. The runtime handoff — agent creation, invocation, telemetry — happens above the substrate and is owned by app teams, exactly as the operating-rhythm table promises.